# TP1 — Détection de drift : PSI & KS test
## Fil rouge : Churn Predictor (Session 5)

**Durée estimée : 1h15**

### 🎯 Objectifs d'apprentissage
À la fin de ce TP vous saurez :
- Distinguer **data drift** vs **concept drift**.
- Implémenter le **PSI** (Population Stability Index) à la main.
- Utiliser le **KS test** (Kolmogorov-Smirnov) pour détecter un drift sur features numériques.
- Simuler 3 scénarios réalistes (no / mild / severe drift) et lire les indicateurs.
- Visualiser les distributions train vs prod simulée pour confirmer.

### 🧠 Pourquoi cette session matter
Votre API tourne depuis 3 mois. Comment vous savez si le modèle continue à bien fonctionner ?

**Le piège classique** : sans monitoring, un modèle peut se dégrader silencieusement pendant des mois. Le marketing change le pricing, un nouveau segment client arrive, une crise économique change les comportements... Toutes ces mutations modifient les distributions des features (= **data drift**) et **le modèle entraîné sur l'ancien monde se trompe sur le nouveau**.


## 1. Théorie en 5 minutes

### Data drift vs concept drift

| Type | Ce qui change | Exemple |
|------|---------------|---------|
| **Data drift** | La distribution des features X | Le marketing augmente les prix → `MonthlyCharges` shift à droite |
| **Concept drift** | La relation entre X et y | Les clients chers churnaient avant, plus maintenant car concurrence absorbe |

⚠️ **Asymétrie cruciale** : pour détecter le **data drift**, on n'a pas besoin de la cible y. Pour détecter le **concept drift**, on en a besoin (et donc d'un délai pour observer le churn réel des clients récents).

→ En prod, on commence **toujours** par monitorer le data drift (pas cher, automatisable, signal précoce).

### PSI (Population Stability Index)

Mesure la divergence entre la distribution **baseline** (entraînement) et la distribution **actuelle** (prod).

$$
\text{PSI} = \sum_{i=1}^{n} (\text{actual}_i - \text{expected}_i) \cdot \ln\left(\frac{\text{actual}_i}{\text{expected}_i}\right)
$$

où `expected_i` et `actual_i` sont les **proportions** dans le bin `i` (numérique : binning equal-width ; catégoriel : par modalité).

**Règles d'or** (industrie crédit/scoring) :
- PSI < 0.10 → **OK**, pas de drift significatif
- 0.10 ≤ PSI < 0.25 → **WARNING**, drift modéré, à surveiller
- PSI ≥ 0.25 → **CRITICAL**, drift significatif, action requise

### KS test (Kolmogorov-Smirnov)

Test statistique non-paramétrique qui mesure la différence maximale entre 2 distributions cumulatives empiriques. Sortie : statistique D ∈ [0, 1] et p-value.

- **PSI** : agrégat numérique simple, lisible (catégoriel possible).
- **KS** : test statistique formel avec p-value (numérique uniquement).

→ En prod, on utilise les **deux** : PSI pour le tableau de bord, KS pour le test rigoureux.


## 2. Setup & rechargement des splits

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42


In [ ]:
X_train_fe = pd.read_csv("data/X_train_fe.csv")
X_test_fe = pd.read_csv("data/X_test_fe.csv")
print(f"Train (baseline): {X_train_fe.shape}")
print(f"Test (source for prod simulations): {X_test_fe.shape}")


## 3. Implémentation du PSI

🎯 **Tâche** : implémentez `psi_numeric` et `psi_categorical`.
- `psi_numeric` : binning equal-width sur la baseline, puis comptage des proportions, puis formule PSI.
- `psi_categorical` : mêmes formule, mais binning = modalités existantes.

In [ ]:
def psi_numeric(expected: np.ndarray, actual: np.ndarray, n_bins: int = 10) -> float:
    """PSI for numeric features. Bin edges defined on `expected`."""
    expected = np.asarray(expected, dtype=float)
    actual = np.asarray(actual, dtype=float)
    
    bin_edges = np.linspace(expected.min(), expected.max(), n_bins + 1)
    bin_edges[0] -= 1e-9
    bin_edges[-1] += 1e-9
    
    expected_counts, _ = np.histogram(expected, bins=bin_edges)
    actual_counts, _ = np.histogram(actual, bins=bin_edges)
    
    expected_pct = expected_counts / max(expected_counts.sum(), 1)
    actual_pct = actual_counts / max(actual_counts.sum(), 1)
    
    eps = 1e-6
    expected_pct = np.where(expected_pct == 0, eps, expected_pct)
    actual_pct = np.where(actual_pct == 0, eps, actual_pct)
    
    return float(np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct)))


def psi_categorical(expected: pd.Series, actual: pd.Series) -> float:
    """PSI for categorical features."""
    cats = sorted(set(expected.unique()) | set(actual.unique()))
    expected_pct = expected.value_counts(normalize=True).reindex(cats, fill_value=0).values
    actual_pct = actual.value_counts(normalize=True).reindex(cats, fill_value=0).values
    eps = 1e-6
    expected_pct = np.where(expected_pct == 0, eps, expected_pct)
    actual_pct = np.where(actual_pct == 0, eps, actual_pct)
    return float(np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct)))


# Smoke test: PSI of a distribution against itself ≈ 0
sanity = psi_numeric(X_train_fe["tenure"].values, X_train_fe["tenure"].values)
print(f"PSI(train, train) = {sanity:.6f}  (should be ~0)")


## 4. Simulation de 3 scénarios de production

On génère 3 datasets "production" en partant du test set, avec différents niveaux de drift :
- **No drift** : bootstrap simple (échantillonnage avec remise)
- **Mild drift** : `MonthlyCharges` × 1.10, `tenure` + 5
- **Severe drift** : `MonthlyCharges` × 1.30, `tenure` − 10, `Contract` forcé à `Month-to-month` à 70 %

In [ ]:
N_PROD = 500
rng = np.random.default_rng(RANDOM_STATE)

def simulate_no_drift(test_df, n=N_PROD):
    return test_df.sample(n=n, replace=True, random_state=RANDOM_STATE).reset_index(drop=True)

def simulate_mild_drift(test_df, n=N_PROD):
    df = test_df.sample(n=n, replace=True, random_state=RANDOM_STATE).reset_index(drop=True)
    df["MonthlyCharges"] = df["MonthlyCharges"] * 1.10
    df["tenure"] = (df["tenure"] + 5).clip(0, 100)
    return df

def simulate_severe_drift(test_df, n=N_PROD):
    df = test_df.sample(n=n, replace=True, random_state=RANDOM_STATE).reset_index(drop=True)
    df["MonthlyCharges"] = df["MonthlyCharges"] * 1.30
    df["tenure"] = (df["tenure"] - 10).clip(0, 100)
    n_force = int(0.7 * len(df))
    df.loc[df.index[:n_force], "Contract"] = "Month-to-month"
    return df

prod_none = simulate_no_drift(X_test_fe)
prod_mild = simulate_mild_drift(X_test_fe)
prod_severe = simulate_severe_drift(X_test_fe)

print(f"prod_none: {prod_none.shape}, MonthlyCharges mean = {prod_none['MonthlyCharges'].mean():.2f}")
print(f"prod_mild: {prod_mild.shape}, MonthlyCharges mean = {prod_mild['MonthlyCharges'].mean():.2f}")
print(f"prod_severe: {prod_severe.shape}, MonthlyCharges mean = {prod_severe['MonthlyCharges'].mean():.2f}")


## 5. Calcul du PSI sur les features critiques

🎯 **Tâche** : pour chaque scénario, calculer le PSI sur les features `MonthlyCharges`, `tenure`, `TotalCharges` (numériques) et `Contract`, `InternetService` (catégorielles).

In [ ]:
NUMERIC = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL = ["Contract", "InternetService", "PaymentMethod"]

def compute_drift_table(baseline, prod_dfs):
    """Build a DataFrame: rows = features, cols = scenarios. Values = PSI."""
    rows = []
    for feat in NUMERIC:
        # TODO: for each scenario in prod_dfs, compute psi_numeric
        row = {"feature": feat, "type": "numeric"}
        for name, df in prod_dfs.items():
            row[name] = ...
        rows.append(row)
    for feat in CATEGORICAL:
        # TODO: for each scenario, compute psi_categorical
        row = {"feature": feat, "type": "categorical"}
        for name, df in prod_dfs.items():
            row[name] = ...
        rows.append(row)
    return pd.DataFrame(rows)


prod_dfs = {
    "no_drift": prod_none,
    "mild_drift": prod_mild,
    "severe_drift": prod_severe,
}
drift_table = compute_drift_table(X_train_fe, prod_dfs)
drift_table.round(4)


### Visualiser le drift table

In [ ]:
heatmap_df = drift_table.set_index("feature")[["no_drift", "mild_drift", "severe_drift"]]

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(heatmap_df, annot=heatmap_df.round(3), fmt=".3f", cmap="RdYlGn_r",
            center=0.10, vmin=0, vmax=0.5, ax=ax, cbar_kws={"label": "PSI"})
ax.set_title("PSI per feature x scenario")
plt.tight_layout()
plt.show()


💡 **Lecture attendue** :
- **no_drift** : tout vert. Le bootstrap reproduit la distribution → PSI proche de 0. C'est notre **calibration** : valide notre implémentation.
- **mild_drift** : `MonthlyCharges` et `tenure` virent au jaune (warning). Les autres features intactes restent vertes.
- **severe_drift** : `MonthlyCharges`, `tenure`, `TotalCharges` (dérivée) ET `Contract` virent au rouge. Le système d'alerte doit déclencher.

🧠 **Réflexe Tech Lead** : la valeur de cet outil n'est pas dans la formule PSI (5 lignes de code). Elle est dans le **système d'alerte** qu'on construit autour : seuil → status → notification → action.

## 6. KS test : la version statistique formelle

PSI est une **métrique** (pas un test). Pour une **décision statistique** (avec p-value), on utilise le **KS test** sur les features numériques.

In [ ]:
def ks_test_table(baseline, prod_dfs, features):
    rows = []
    for feat in features:
        row = {"feature": feat}
        for name, df in prod_dfs.items():
            stat, pval = stats.ks_2samp(baseline[feat], df[feat])
            row[f"{name}_D"] = stat
            row[f"{name}_pval"] = pval
        rows.append(row)
    return pd.DataFrame(rows)


ks_table = ks_test_table(X_train_fe, prod_dfs, NUMERIC)
print("KS test results (D statistic + p-value):")
ks_table.round(5)


💡 **Lecture** : 
- p-value < 0.05 → on rejette l'hypothèse "même distribution". Drift confirmé statistiquement.
- Pour `no_drift`, p-value typiquement > 0.05 → pas de drift détecté.
- Pour `severe_drift`, p-value très petite (souvent < 1e-50) → drift massif.

⚠️ **Limite du KS test** : avec beaucoup de données, **n'importe quel petit écart devient significatif** (la p-value tend vers 0). C'est pourquoi en prod, on regarde **PSI ET KS** : la magnitude (PSI) ET la significativité (KS).


## 7. Visualiser les distributions

Un PSI seul est abstrait. On confirme visuellement avec des histogrammes train vs prod.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# MonthlyCharges
axes[0, 0].hist(X_train_fe["MonthlyCharges"], bins=30, alpha=0.6, label="train (baseline)", density=True)
axes[0, 0].hist(prod_severe["MonthlyCharges"], bins=30, alpha=0.6, label="prod (severe)", density=True)
axes[0, 0].set_title("MonthlyCharges — train vs severe drift")
axes[0, 0].legend()

# tenure
axes[0, 1].hist(X_train_fe["tenure"], bins=30, alpha=0.6, label="train", density=True)
axes[0, 1].hist(prod_severe["tenure"], bins=30, alpha=0.6, label="prod (severe)", density=True)
axes[0, 1].set_title("tenure — train vs severe drift")
axes[0, 1].legend()

# Contract
def cat_pct(s, name):
    return s.value_counts(normalize=True).rename(name)
contract_compare = pd.DataFrame([
    cat_pct(X_train_fe["Contract"], "train"),
    cat_pct(prod_severe["Contract"], "prod (severe)"),
]).T
contract_compare.plot(kind="bar", ax=axes[1, 0])
axes[1, 0].set_title("Contract distribution — train vs severe drift")
axes[1, 0].set_ylabel("Proportion")
axes[1, 0].tick_params(axis="x", rotation=20)

# Mild drift visualization
axes[1, 1].hist(X_train_fe["MonthlyCharges"], bins=30, alpha=0.6, label="train", density=True)
axes[1, 1].hist(prod_mild["MonthlyCharges"], bins=30, alpha=0.6, label="prod (mild)", density=True)
axes[1, 1].set_title("MonthlyCharges — train vs MILD drift (subtler)")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


## 8. Bilan TP1

Vous avez :
- ✅ Implémenté le PSI à la main (numérique + catégoriel)
- ✅ Simulé 3 niveaux de drift réalistes
- ✅ Construit un tableau de drift et son visuel "feu tricolore"
- ✅ Croisé PSI et KS pour une décision robuste

🎯 **Question** : maintenant qu'on sait **détecter** le drift, comment on l'**intègre dans une API en prod** ? C'est le TP2.

🛑 **Pause de 15 minutes**.